# Comprehensive PyTorch Autograd Tutorial

| Date | Who | Mail |  What |
| ---  | --- | ---  | ---   |     
| Nov 24, 2025 | Diego Andrés Alvarez Marín | <daalvarez@unal.edu.co>  | Initial code created using Gemini 3.0 |
| Nov 24, 2025 | Diego Andrés Alvarez Marín | <daalvarez@unal.edu.co>  | Improvements to code and comments |

This notebook explores **automatic differentiation** in PyTorch. We will analyze the computational graph, gradient computation, and advanced derivative manipulation using the test function:

$$f(x_1, x_2) = \ln(x_1) + x_1 x_2 - \sin(x_2)$$

## 1. Setup and Basic Definition

First, we import the necessary libraries and define our variables. To compute gradients, we must set `requires_grad=True` on our input tensors.

In [ ]:
import torch

# Define inputs
# x1 must be > 0 for ln(x1)
x1 = torch.tensor([2.0], requires_grad=True)
x2 = torch.tensor([5.0], requires_grad=True)

print(f"x1: {x1}, x1: {x1.item()}, requires_grad: {x1.requires_grad}")
print(f"x2: {x2}, x2: {x2.item()}, requires_grad: {x2.requires_grad}")

x1: tensor([2.], requires_grad=True), x1: 2.0, requires_grad: True
x2: tensor([5.], requires_grad=True), x2: 5.0, requires_grad: True


-----

## 2\. The Computational Graph & Forward Pass

When we perform operations on tensors with `requires_grad=True`, PyTorch builds a Dynamic Computational Graph.

  * **`grad_fn`**: References the function that created the tensor (e.g., `AddBackward`, `SinBackward`, `SubBackward`). This links the graph together for backpropagation.

<!-- end list -->

In [ ]:
# Define the function
# f(x1, x2) = ln(x1) + x1*x2 - sin(x2)
y = torch.log(x1) + x1*x2 - torch.sin(x2)

print(f"Output y: {y}, y: {y.item():.4f}")

# Inspecting the graph via grad_fn
print(f"y.grad_fn: {y.grad_fn}")
# Expected: SubBackward0 represents the last operation: subtraction

Output y: tensor([11.6521], grad_fn=<SubBackward0>), y: 11.6521
y.grad_fn: <SubBackward0 object at 0x7d76ebbb9360>


-----

## 3\. Basic Backpropagation

### `.backward()` and `.grad`

Calling `.backward()` on a scalar output computes the gradient of that output with respect to leaves (inputs) in the graph. The results are stored in the `.grad` attribute of the input tensors.

$$
\frac{\partial f}{\partial x_1} = \frac{1}{x_1} + x_2 \quad \text{and} \quad \frac{\partial f}{\partial x_2} = x_1 - \cos(x_2)
$$

In [ ]:
# 1. Trigger backpropagation
y.backward()

# 2. Access gradients
dx1 = x1.grad
dx2 = x2.grad

print(f"df/dx1 (PyTorch): {dx1.item():.4f}")
print(f"df/dx2 (PyTorch): {dx2.item():.4f}")

# Verification
manual_dx1 = (1/x1.item()) + x2.item()
manual_dx2 = x1.item() - torch.cos(x2).item()
print(f"df/dx1 (Manual):  {manual_dx1:.4f}")
print(f"df/dx2 (Manual):  {manual_dx2:.4f}")

df/dx1 (PyTorch): 5.5000
df/dx2 (PyTorch): 1.7163
df/dx1 (Manual):  5.5000
df/dx2 (Manual):  1.7163


### `.grad.zero_()`

**Crucial Note:** Gradients in PyTorch accumulate (add up) by default. You must clear them before a new optimization step.

In [ ]:
print(f"Grad before zeroing: {x1.grad.item()}")

# Clear gradients
x1.grad.zero_() # The _ at the end of the command denotes an in-place operation.
x2.grad.zero_()
x2.grad.zero_()

print(f"Grad after zeroing: {x1.grad.item()}")

Grad before zeroing: 5.5
Grad after zeroing: 0.0


-----

## 4\. Controlling Gradient Tracking

There are several ways to stop PyTorch from tracking history, which saves memory during inference.

### `torch.no_grad()`, `torch.enable_grad()`, `torch.set_grad_enabled()`

In [ ]:
# 1. torch.no_grad() - Standard context manager for inference
with torch.no_grad():
    y_inference = torch.log(x1) + (x1 * x2) - torch.sin(x2)
    print(f"Inside no_grad, requires_grad: {y_inference.requires_grad}") # False

# 2. torch.set_grad_enabled(bool) - Programmatic toggle
torch.set_grad_enabled(False)
y_disabled = x1 * x2
print(f"With set_grad_enabled(False): {y_disabled.requires_grad}") # False
torch.set_grad_enabled(True) # Turn back on

# 3. torch.enable_grad() - Using gradients inside a no_grad block
with torch.no_grad():
    with torch.enable_grad():
        y_enabled = x1 * x2
        print(f"Inside enable_grad nested in no_grad: {y_enabled.requires_grad}") # True

Inside no_grad, requires_grad: False
With set_grad_enabled(False): False
Inside enable_grad nested in no_grad: True


### `.detach()`

This creates a new tensor that shares storage with the original but is detached from the computational graph.

In [ ]:
y_live = torch.log(x1)
y_detached = y_live.detach()

print(f"Original requires_grad: {y_live.requires_grad}")     # True
print(f"Detached requires_grad: {y_detached.requires_grad}") # False

Original requires_grad: True
Detached requires_grad: False


-----

## 5\. Advanced Autograd Mechanics

### `create_graph`

**`create_graph=True`**: Computes the derivative of the derivative (needed for higher-order gradients).

In [ ]:
x = torch.tensor(4.0, requires_grad=True)
y = x ** 3

# First derivative: 3x^2 = 3*16 = 48
# create_graph=True allows us to differentiate 'grad_1' later
grad_1 = torch.autograd.grad(outputs=y, inputs=x, create_graph=True)[0]
print(f"1st Derivative: {grad_1.item()}")

# Second derivative: 6x = 6*4 = 24
grad_2 = torch.autograd.grad(outputs=grad_1, inputs=x)[0]
print(f"2nd Derivative: {grad_2.item()}")

1st Derivative: 48.0
2nd Derivative: 24.0


### `retain_graph`

By default, when you call `.backward()`, PyTorch **destroys** the computational graph to free up memory. If you try to use that graph again, you get an error.

**When do you need `retain_graph=True`?**
You need it if you want to call `.backward()` **twice** on the same graph (for example, if you have two different loss functions that share the same input calculation).

#### Scenario A: The Crash (Without `retain_graph`)

In [ ]:
x = torch.tensor([2.0], requires_grad=True)
output = x ** 2  # The graph is built here

loss1 = output.sum()
loss2 = output.mean()

# First backward pass
loss1.backward()
print("First backward successful")

# Second backward pass FAILS because the graph was deleted above
try:
    loss2.backward()
except RuntimeError as e:
    print("\nERROR CAUGHT:", e)

First backward successful

ERROR CAUGHT: Trying to backward through the graph a second time (or directly access saved tensors after they have already been freed). Saved intermediate values of the graph are freed when you call .backward() or autograd.grad(). Specify retain_graph=True if you need to backward through the graph a second time or if you need to access saved tensors after calling backward.


#### Scenario B: The Fix (With `retain_graph`)

In [ ]:
# Reset gradients and inputs
x = torch.tensor([2.0], requires_grad=True)
output = x ** 2

loss1 = output.sum()
loss2 = output.mean()

# First backward pass: We tell PyTorch "Don't delete the graph yet!"
loss1.backward(retain_graph=True)
print(f"Grad after Loss 1: {x.grad}") # 2x = 4

# Second backward pass: Now we can use the graph again.
# Note: Gradients accumulate (add up).
loss2.backward()
print(f"Grad after Loss 2: {x.grad}") # Previous 4 + (2x) = 8

Grad after Loss 1: tensor([4.])
Grad after Loss 2: tensor([8.])


### `torch.autograd.grad` vs `.backward()`

`.backward()` accumulates into `.grad`. `torch.autograd.grad` computes and returns the gradients immediately without accumulation.

In [ ]:
x1.grad.zero_()
x2.grad.zero_()
y = torch.log(x1) + (x1 * x2) - torch.sin(x2)

# Calculate gradients directly without storing them in x1.grad
grads = torch.autograd.grad(outputs=y, inputs=[x1, x2])
print(f"Direct grads: {grads}")
print(f"x1.grad is still: {x1.grad}") # None or 0

Direct grads: (tensor([5.5000]), tensor([1.7163]))
x1.grad is still: tensor([0.])


-----

## 6\. Anomaly Detection

Debugging NaNs in backward passes can be difficult. `set_detect_anomaly` provides a traceback to the specific operation that created the bad value.

In [ ]:
torch.autograd.set_detect_anomaly(True)

try:
    x_bad = torch.tensor([5.0], requires_grad=True)
    # Force a NaN by taking sqrt of a negative number
    # We use a small operation (x - 6) to ensure 5 - 6 = -1
    val = x_bad - 6.0
    y_bad = torch.sqrt(val)
    y_bad.backward()
except RuntimeError as e:
    print("Anomaly Successfully Detected!")
    print(e)

# Turn it off when done
torch.autograd.set_detect_anomaly(False)

Anomaly Successfully Detected!
Function 'SqrtBackward0' returned nan values in its 0th output.


-----

## 7. Computing Jacobians, Hessians, and Vector products

First, let's define our function as a pure Python function:

In [ ]:
# 1. Define the function to accept unwrapped arguments
def func(x1, x2):
    return torch.log(x1) + (x1 * x2) - torch.sin(x2)

# 2. Define inputs
inputs = (torch.tensor([2.0]), torch.tensor([5.0]))

### Jacobian and Hessian

  * **Jacobian:** First-order partial derivatives.
  * **Hessian:** Second-order partial derivatives.

<!-- end list -->

In [ ]:
# Jacobian: Returns nested tuple of gradients
# result structure: ((df/dx1), (df/dx2))
jac = torch.autograd.functional.jacobian(func, inputs)
print("Jacobian (df/dx1, df/dx2):")
print(jac)

# Hessian: Returns nested tuple of 2nd derivatives
hess = torch.autograd.functional.hessian(func, inputs)
print("\nHessian Matrix (d2f/dx1^2,  d2f/dx1dx2):")
print(hess[0])
print("Hessian Matrix (d2f/dx2dx1, d2f/dx2^2):")
print(hess[1])

Jacobian (df/dx1, df/dx2):
(tensor([[5.5000]]), tensor([[1.7163]]))

Hessian Matrix (d2f/dx1^2,  d2f/dx1dx2):
(tensor([[-0.2500]]), tensor([[1.]]))
Hessian Matrix (d2f/dx2dx1, d2f/dx2^2):
(tensor([[1.]]), tensor([[-0.9589]]))


### JVP and VJP

These compute product operations without necessarily instantiating the full Jacobian matrix (memory efficient).

  * **JVP (Jacobian-Vector Product):** It uses **forward mode** AD. Given a vector $v$, computes $J v$.
  * **VJP (Vector-Jacobian Product):** It uses **reverse mode** AD(Backpropagation). Given a vector $v$, computes $v^T J$.


In [ ]:
import torch.autograd.functional as functional

# JVP: "If inputs change by vector v, how much does output change?"
# v must match the shape of inputs (one for x1, one for x2)
v = (torch.tensor([1.0]), torch.tensor([1.0]))
jvp_out = torch.autograd.functional.jvp(func, inputs, v=v)

# Returns: (Function Output, Gradient value based on v)
print(f"\nJVP Output: {jvp_out}")

v = torch.tensor([1.0])
vjp_out = torch.autograd.functional.vjp(func, inputs, v=v)
print(f"VJP Output: {vjp_out}")


JVP Output: (tensor([11.6521]), tensor([7.2163]))
VJP Output: (tensor([11.6521]), (tensor([5.5000]), tensor([1.7163])))
